# NB02: Bridge Tables & RAST Processing

**Purpose**: Build the three bridge tables that connect evidence channels to mass-balanced ModelSEED reactions,
and process the RAST validation set.

**Bridge tables**:
1. EC → ModelSEED reaction (from user-provided lookup, filtered to status=OK)
2. KEGG R-number → ModelSEED reaction (from `reaction.abbreviation`)
3. MetaCyc ID → ModelSEED reaction (from `reaction.abbreviation`)

**RAST processing**: Extract EC numbers from 84.5M RAST annotation strings.

**Critical**: All bridges filter to mass-balanced reactions only (`status = 'OK'`, 34,343 reactions).
Unbalanced coverage is reported separately as a ceiling.

**Requires**: BERDL JupyterHub (Spark session)

In [1]:
import os, re
import pandas as pd

try:
    from berdl_notebook_utils.setup_spark_session import get_spark_session
except ImportError:
    from get_spark_session import get_spark_session

spark = get_spark_session()
spark.sql("SET spark.sql.autoBroadcastJoinThreshold = -1")

DATA_DIR = '../data'
USER_DIR = '../user_data'
os.makedirs(DATA_DIR, exist_ok=True)

print('Spark session ready.')

Spark session ready.


## 1. Load Reaction Table & Mass-Balance Filter

Load from cache (NB01) and establish the balanced reaction set.

In [2]:
reactions = pd.read_csv(f'{DATA_DIR}/reactions_all.tsv', sep='\t')
print(f'Total reactions: {len(reactions):,}')

balanced = reactions[reactions['status'] == 'OK'].copy()
unbalanced = reactions[reactions['status'] != 'OK'].copy()
print(f'Mass-balanced (OK): {len(balanced):,}')
print(f'Unbalanced/other:   {len(unbalanced):,}')

balanced['rxn_bare'] = balanced['id'].str.replace('seed.reaction:', '', regex=False)
balanced_ids = set(balanced['rxn_bare'])
print(f'Unique balanced bare IDs: {len(balanced_ids):,}')

Total reactions: 56,012
Mass-balanced (OK): 34,343
Unbalanced/other:   21,669
Unique balanced bare IDs: 34,343


## 2. EC → Reaction Bridge

User-provided `Unique_ModelSEED_Reaction_ECs.txt` maps EC numbers to bare ModelSEED reaction IDs.
Filter to balanced reactions only; report unbalanced ceiling.

In [3]:
ec_raw = pd.read_csv(f'{USER_DIR}/Unique_ModelSEED_Reaction_ECs.txt', sep='\t')
ec_raw.columns = ['rxn_bare', 'ec', 'source']
print(f'Raw EC-reaction mappings: {len(ec_raw):,}')
print(f'  Unique reactions: {ec_raw.rxn_bare.nunique():,}')
print(f'  Unique ECs:       {ec_raw.ec.nunique():,}')

ec_balanced = ec_raw[ec_raw['rxn_bare'].isin(balanced_ids)].copy()
ec_unbalanced = ec_raw[~ec_raw['rxn_bare'].isin(balanced_ids)]

print(f'\nAfter mass-balance filter (status=OK):')
print(f'  Balanced mappings:   {len(ec_balanced):,}')
print(f'  Balanced reactions:  {ec_balanced.rxn_bare.nunique():,}')
print(f'  Balanced ECs:        {ec_balanced.ec.nunique():,}')
print(f'\nUnbalanced ceiling (excluded):')
print(f'  Unbalanced mappings: {len(ec_unbalanced):,}')
print(f'  Unbalanced reactions: {ec_unbalanced.rxn_bare.nunique():,}')
print(f'  ECs only in unbalanced: {len(set(ec_unbalanced.ec) - set(ec_balanced.ec)):,}')

Raw EC-reaction mappings: 30,789
  Unique reactions: 25,756
  Unique ECs:       7,350

After mass-balance filter (status=OK):
  Balanced mappings:   22,823
  Balanced reactions:  18,660
  Balanced ECs:        6,100

Unbalanced ceiling (excluded):
  Unbalanced mappings: 7,966
  Unbalanced reactions: 7,096
  ECs only in unbalanced: 1,250


In [4]:
ec_balanced['rxn_id'] = 'seed.reaction:' + ec_balanced['rxn_bare']
ec_bridge = ec_balanced[['ec', 'rxn_id', 'rxn_bare']].copy()
ec_bridge.to_parquet(f'{DATA_DIR}/ec_to_reaction.parquet', index=False)
print(f'Saved {len(ec_bridge):,} EC-reaction mappings to ec_to_reaction.parquet')

print(f'\nEC multiplicity (reactions per EC):')
mult = ec_bridge.groupby('ec').size()
print(f'  1 reaction:    {(mult == 1).sum():,} ECs')
print(f'  2-5 reactions: {((mult >= 2) & (mult <= 5)).sum():,} ECs')
print(f'  6+ reactions:  {(mult >= 6).sum():,} ECs')
print(f'  Max:           {mult.max()} reactions for EC {mult.idxmax()}')

Saved 22,823 EC-reaction mappings to ec_to_reaction.parquet

EC multiplicity (reactions per EC):
  1 reaction:    3,113 ECs
  2-5 reactions: 2,213 ECs
  6+ reactions:  774 ECs
  Max:           293 reactions for EC 2.4.1.-


## 3. KEGG R-number → Reaction Bridge

Extract KEGG R-numbers from `reaction.abbreviation` for balanced reactions.
Pattern: abbreviation starts with `R` followed by 5 digits.

In [5]:
kegg_pattern = re.compile(r'^(R\d{5})\b')

kegg_rows = []
for _, row in balanced.iterrows():
    abbrev = row.get('abbreviation', '')
    if pd.isna(abbrev) or abbrev == '':
        continue
    m = kegg_pattern.match(str(abbrev))
    if m:
        kegg_rows.append({
            'kegg_reaction': m.group(1),
            'rxn_id': row['id'],
            'rxn_bare': row['rxn_bare'],
        })

kegg_bridge = pd.DataFrame(kegg_rows)
print(f'KEGG R-number to balanced reaction mappings: {len(kegg_bridge):,}')
print(f'  Unique KEGG R-numbers: {kegg_bridge.kegg_reaction.nunique():,}')
print(f'  Unique reactions:      {kegg_bridge.rxn_id.nunique():,}')

kegg_mult = kegg_bridge.groupby('kegg_reaction').size()
print(f'\nKEGG multiplicity:')
print(f'  1 reaction:    {(kegg_mult == 1).sum():,}')
print(f'  2+ reactions:  {(kegg_mult >= 2).sum():,}')

kegg_bridge.to_parquet(f'{DATA_DIR}/kegg_to_reaction.parquet', index=False)
print(f'\nSaved to kegg_to_reaction.parquet')

kegg_unbal = sum(1 for _, row in unbalanced.iterrows()
                 if pd.notna(row.get('abbreviation', ''))
                 and kegg_pattern.match(str(row.get('abbreviation', ''))))
print(f'\nUnbalanced ceiling: {kegg_unbal:,} additional KEGG mappings excluded')

KEGG R-number to balanced reaction mappings: 6,851
  Unique KEGG R-numbers: 6,354
  Unique reactions:      6,851

KEGG multiplicity:
  1 reaction:    6,010
  2+ reactions:  344

Saved to kegg_to_reaction.parquet



Unbalanced ceiling: 2,402 additional KEGG mappings excluded


## 4. MetaCyc → Reaction Bridge

Extract MetaCyc reaction IDs from `reaction.abbreviation` for balanced reactions.
Pattern: abbreviation contains `-RXN` (e.g., `ARG-OXIDATION-RXN.c`, `PREPHENATEDEHYDRAT-RXN`).
Strip trailing `.c` compartment suffixes.

In [6]:
metacyc_rows = []
for _, row in balanced.iterrows():
    abbrev = row.get('abbreviation', '')
    if pd.isna(abbrev) or abbrev == '':
        continue
    abbrev_str = str(abbrev)
    if '-RXN' in abbrev_str:
        metacyc_id = re.sub(r'\.[a-z]$', '', abbrev_str)
        metacyc_rows.append({
            'metacyc_reaction': metacyc_id,
            'rxn_id': row['id'],
            'rxn_bare': row['rxn_bare'],
            'abbreviation_raw': abbrev_str,
        })

metacyc_bridge = pd.DataFrame(metacyc_rows)
print(f'MetaCyc to balanced reaction mappings: {len(metacyc_bridge):,}')
print(f'  Unique MetaCyc IDs: {metacyc_bridge.metacyc_reaction.nunique():,}')
print(f'  Unique reactions:   {metacyc_bridge.rxn_id.nunique():,}')

metacyc_bridge_out = metacyc_bridge[['metacyc_reaction', 'rxn_id', 'rxn_bare']].copy()
metacyc_bridge_out.to_parquet(f'{DATA_DIR}/metacyc_to_reaction.parquet', index=False)
print(f'Saved to metacyc_to_reaction.parquet')

metacyc_unbal = sum(1 for _, row in unbalanced.iterrows()
                    if pd.notna(row.get('abbreviation', ''))
                    and '-RXN' in str(row.get('abbreviation', '')))
print(f'\nUnbalanced ceiling: {metacyc_unbal:,} additional MetaCyc mappings excluded')

MetaCyc to balanced reaction mappings: 5,039
  Unique MetaCyc IDs: 4,480
  Unique reactions:   5,039
Saved to metacyc_to_reaction.parquet



Unbalanced ceiling: 2,273 additional MetaCyc mappings excluded


## 5. Bridge Table Summary

How many balanced reactions are reachable through each bridge?

In [7]:
ec_rxns = set(ec_bridge['rxn_bare'])
kegg_rxns = set(kegg_bridge['rxn_bare'])
metacyc_rxns = set(metacyc_bridge_out['rxn_bare'])
any_bridge = ec_rxns | kegg_rxns | metacyc_rxns

print(f'Balanced reactions reachable by bridge:')
print(f'  EC bridge only:      {len(ec_rxns):,} / {len(balanced_ids):,} ({100*len(ec_rxns)/len(balanced_ids):.1f}%)')
print(f'  KEGG bridge only:    {len(kegg_rxns):,} / {len(balanced_ids):,} ({100*len(kegg_rxns)/len(balanced_ids):.1f}%)')
print(f'  MetaCyc bridge only: {len(metacyc_rxns):,} / {len(balanced_ids):,} ({100*len(metacyc_rxns)/len(balanced_ids):.1f}%)')
print(f'  Any bridge:          {len(any_bridge):,} / {len(balanced_ids):,} ({100*len(any_bridge)/len(balanced_ids):.1f}%)')
print(f'  No bridge:           {len(balanced_ids - any_bridge):,} reactions unreachable')

print(f'\nOverlap:')
print(f'  EC & KEGG:           {len(ec_rxns & kegg_rxns):,}')
print(f'  EC & MetaCyc:        {len(ec_rxns & metacyc_rxns):,}')
print(f'  KEGG & MetaCyc:      {len(kegg_rxns & metacyc_rxns):,}')
print(f'  All three:           {len(ec_rxns & kegg_rxns & metacyc_rxns):,}')

Balanced reactions reachable by bridge:
  EC bridge only:      18,660 / 34,343 (54.3%)
  KEGG bridge only:    6,851 / 34,343 (19.9%)
  MetaCyc bridge only: 5,039 / 34,343 (14.7%)
  Any bridge:          20,227 / 34,343 (58.9%)
  No bridge:           14,116 reactions unreachable

Overlap:
  EC & KEGG:           6,462
  EC & MetaCyc:        3,861
  KEGG & MetaCyc:      0
  All three:           0


## 6. RAST Validation Set Processing

Parse `uniprot_rast_filtered_mapping.tsv.gz` (84.5M rows) to extract EC numbers from RAST annotation strings.
Pattern: `(EC x.x.x.x)` embedded in annotation text.

Use Spark to handle the 84.5M rows efficiently.

In [8]:
from pyspark.sql import functions as F
import gzip

rast_path = os.path.join(USER_DIR, 'uniprot_rast_filtered_mapping.tsv.gz')

ec_re = re.compile(r'\(EC ([0-9]+\.[0-9\-]+\.[0-9\-]+\.[0-9\-]+)\)')

pairs = set()
total_rows = 0
ec_rows = 0
multi_ec_rows = 0

with gzip.open(rast_path, 'rt') as f:
    header = f.readline()
    for line in f:
        total_rows += 1
        parts = line.rstrip('\n').split('\t', 1)
        if len(parts) < 2:
            continue
        protein_id, annotation = parts
        matches = ec_re.findall(annotation)
        if matches:
            ec_rows += 1
            if len(matches) > 1:
                multi_ec_rows += 1
            for ec in matches:
                pairs.add((protein_id, ec))

        if total_rows % 10_000_000 == 0:
            print(f'  ...processed {total_rows:,} rows, {len(pairs):,} pairs so far')

print(f'Total rows: {total_rows:,}')
print(f'Rows with EC: {ec_rows:,}')
print(f'Rows with multiple ECs: {multi_ec_rows:,}')
print(f'Distinct protein-EC pairs: {len(pairs):,}')

  ...processed 10,000,000 rows, 4,090,834 pairs so far


  ...processed 20,000,000 rows, 8,037,321 pairs so far


  ...processed 30,000,000 rows, 11,916,224 pairs so far


  ...processed 40,000,000 rows, 15,835,973 pairs so far


  ...processed 50,000,000 rows, 19,579,483 pairs so far


  ...processed 60,000,000 rows, 23,345,037 pairs so far


  ...processed 70,000,000 rows, 27,052,090 pairs so far


  ...processed 80,000,000 rows, 30,790,275 pairs so far


Total rows: 84,476,981
Rows with EC: 32,146,795
Rows with multiple ECs: 2,256,635
Distinct protein-EC pairs: 32,420,974


In [9]:
rast_df = pd.DataFrame(list(pairs), columns=['protein_id', 'ec'])
print(f'Unique proteins: {rast_df.protein_id.nunique():,}')
print(f'Unique ECs: {rast_df.ec.nunique():,}')

def ec_level(ec):
    if '-' not in ec:
        return '4-digit (complete)'
    if ec.endswith('.-') and not ec.replace('.-', '', 1).endswith('.-'):
        return '3-digit (x.x.x.-)'
    if ec.count('-') == 2:
        return '2-digit (x.x.-.-)'
    return '1-digit (x.-.-.-)'

rast_df['level'] = rast_df['ec'].apply(ec_level)
print(f'\nEC level distribution:')
level_stats = rast_df.groupby('level').agg(
    n_rows=('ec', 'size'),
    n_ecs=('ec', 'nunique'),
)
print(level_stats.to_string())

Unique proteins: 30,363,667


Unique ECs: 2,439



EC level distribution:


                      n_rows  n_ecs
level                              
1-digit (x.-.-.-)      57350      6
2-digit (x.x.-.-)     106417     16
3-digit (x.x.x.-)     713069     81
4-digit (complete)  31544138   2336


In [10]:
rast_out = rast_df[['protein_id', 'ec']].copy()
rast_out_count = len(rast_out)

rast_out.to_parquet(f'{DATA_DIR}/rast_protein_ec.parquet', index=False)
print(f'Saved {rast_out_count:,} protein-EC pairs to rast_protein_ec.parquet')

Saved 32,420,974 protein-EC pairs to rast_protein_ec.parquet


In [11]:
bridge_ecs = set(ec_bridge['ec'])
rast_ecs = set(rast_out['ec'])
rast_ecs_in_bridge = rast_ecs & bridge_ecs

rast_proteins_reachable = rast_out[rast_out['ec'].isin(rast_ecs_in_bridge)].protein_id.nunique()
rast_reactions_reachable = ec_bridge[ec_bridge['ec'].isin(rast_ecs_in_bridge)].rxn_bare.nunique()

print(f'RAST proteins reaching balanced reactions via EC bridge:')
print(f'  RAST ECs matching bridge: {len(rast_ecs_in_bridge):,} / {len(rast_ecs):,} ({100*len(rast_ecs_in_bridge)/len(rast_ecs):.1f}%)')
print(f'  RAST ECs NOT in bridge:   {len(rast_ecs - bridge_ecs):,}')
print(f'  Proteins reachable:       {rast_proteins_reachable:,} / {rast_out.protein_id.nunique():,}')
print(f'  Reactions reachable:      {rast_reactions_reachable:,} / {len(balanced_ids):,}')

RAST proteins reaching balanced reactions via EC bridge:
  RAST ECs matching bridge: 2,135 / 2,439 (87.5%)
  RAST ECs NOT in bridge:   304


  Proteins reachable:       26,710,795 / 30,363,667
  Reactions reachable:      10,730 / 34,343


## 7. Summary

In [12]:
print('=' * 60)
print('NB02 BRIDGE TABLE SUMMARY')
print('=' * 60)
print(f'\nTarget: {len(balanced_ids):,} mass-balanced reactions (status=OK)')
print(f'\nBridge tables built (all filtered to balanced only):')
print(f'  1. ec_to_reaction.parquet      {len(ec_bridge):,} EC-reaction mappings')
print(f'  2. kegg_to_reaction.parquet     {len(kegg_bridge):,} KEGG-reaction mappings')
print(f'  3. metacyc_to_reaction.parquet  {len(metacyc_bridge_out):,} MetaCyc-reaction mappings')
print(f'  4. rast_protein_ec.parquet      {rast_out_count:,} protein-EC pairs')
print(f'\nReachable balanced reactions:')
print(f'  Via EC:      {len(ec_rxns):,} ({100*len(ec_rxns)/len(balanced_ids):.1f}%)')
print(f'  Via KEGG:    {len(kegg_rxns):,} ({100*len(kegg_rxns)/len(balanced_ids):.1f}%)')
print(f'  Via MetaCyc: {len(metacyc_rxns):,} ({100*len(metacyc_rxns)/len(balanced_ids):.1f}%)')
print(f'  Any bridge:  {len(any_bridge):,} ({100*len(any_bridge)/len(balanced_ids):.1f}%)')
print(f'\nNext: NB03 -- use these bridges with UniProt-native evidence (Tier 1)')

NB02 BRIDGE TABLE SUMMARY

Target: 34,343 mass-balanced reactions (status=OK)

Bridge tables built (all filtered to balanced only):
  1. ec_to_reaction.parquet      22,823 EC-reaction mappings
  2. kegg_to_reaction.parquet     6,851 KEGG-reaction mappings
  3. metacyc_to_reaction.parquet  5,039 MetaCyc-reaction mappings
  4. rast_protein_ec.parquet      32,420,974 protein-EC pairs

Reachable balanced reactions:
  Via EC:      18,660 (54.3%)
  Via KEGG:    6,851 (19.9%)
  Via MetaCyc: 5,039 (14.7%)
  Any bridge:  20,227 (58.9%)

Next: NB03 -- use these bridges with UniProt-native evidence (Tier 1)
